# HW3 Problem 2

In [ ]:
#Since Python versions higher than 3.12 cause errors, I switched to version 2026.07 in Colab.！！！
%pip install "transformers==4.53.2" "accelerate>=0.26" "sentencepiece"
%pip install "git+https://github.com/structuredllm/syncode.git@v0.4.16"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 84.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 48.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 86.2 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.23.0
    Uninstalling huggingface_hub-1.23.0:
      Successfully uninstalled huggingface_hub-1.23.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.13.1
    Uninstalling transformers-5.13.1:
      Successfully uninstalled transformers-5.13.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the

In [ ]:
import json
import math
import operator
import re

from syncode import Syncode

syn_llm = Syncode(
    model="microsoft/phi-2",
    grammar="json",
    mode="grammar_strict",
    parse_output_only=True,
    max_new_tokens=96,
    do_sample=False,
)
print("ready")

[2026-10-27 06:03:36,160-root] - Loading model microsoft/phi-2 with device:cuda, device_map:auto, torch_dtype:torch.bfloat16


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/735 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

[2026-10-27 06:04:59,600-syncode.mask_store.mask_store] - Using cache: True and fsm path cache/mask_stores/CodeGenTokenizerFast/grammar_strict_3880754916_50257.pkl exist: False
[2026-10-27 06:04:59,601-syncode.mask_store.mask_store] - Creating mask store for CodeGenTokenizerFast and json, may take more than 10 minutes. Caching at /content/cache/mask_stores/CodeGenTokenizerFast/grammar_strict_3880754916_50257.pkl.
[2026-10-27 06:05:00,490-syncode.mask_store.fsm_set] - 13 FSMs with 62 states initialized in 0.03 seconds
[2026-10-27 06:05:00,491-syncode.mask_store.mask_store] - Ignore whitespace is True
[2026-10-27 06:05:00,494-syncode.mask_store.mask_store] - Number of 2 length terminal sequences reduced from 169 to 65


100%|██████████| 62/62 [00:18<00:00,  3.31it/s]


[2026-10-27 06:05:19,467-syncode.mask_store.mask_store] - Time taken to create mask store: 19.82 seconds
ready


## Provide a prompt

In [ ]:
EXTRACTION_PREFIX = """You extract data from arithmetic word problems.
Given one query involving a single binary operation, output ONLY one JSON object:
{"operands": [first_number, second_number], "operator": "+"}
Use JSON numbers (not strings) for operands, preserving the operand order from the mathematical expression.
The operator must be exactly one of "+", "-", "*", "/".
Translate plus/added/sum -> +, minus/subtracted/difference -> -, multiplied/times/product -> *, divided/quotient -> /.
Important: for "A subtracted from B", B is first; for "A divided into B", B is first.
Do NOT calculate the result. Do NOT add explanations or markdown.

Query: What is 327. multiplied by 11.0?
Extraction: {"operands": [327.0, 11.0], "operator": "*"}

Query: What is 45.1 plus 23.54?
Extraction: {"operands": [45.1, 23.54], "operator": "+"}

Query: What is 120.4 divided by 4.0?
Extraction: {"operands": [120.4, 4.0], "operator": "/"}

Query: Subtract 8.5 from 20.0.
Extraction: {"operands": [20.0, 8.5], "operator": "-"}

"""

def _build_prompt(textual_query):
    return EXTRACTION_PREFIX + "Query: " + textual_query.strip() + "\nExtraction: "

## Main Functions

In [ ]:
OPERATIONS = {
    "+": operator.add,
    "-": operator.sub,
    "*": operator.mul,
    "/": operator.truediv,
}


def _parse_extraction(raw_output):

    output = raw_output.strip()
    output = output.replace("\\\\n", "\n").replace("\\n", "\n").replace("\\t", " ")
    output = output.translate(str.maketrans({
        "“": '"', "”": '"', "„": '"', "‘": "'", "’": "'",
        "−": "-", "×": "*", "÷": "/",
    }))

    decoder = json.JSONDecoder()
    candidates = []
    try:
        value, _ = decoder.raw_decode(output)
        candidates.append(value)
    except json.JSONDecodeError:
        pass
    for match in re.finditer(r"\{", output):
        try:
            value, _ = decoder.raw_decode(output[match.start():])
            candidates.append(value)
        except json.JSONDecodeError:
            continue

    for obj in candidates:
        if not isinstance(obj, dict):
            continue
        values = obj.get("operands")
        symbol = obj.get("operator")
        if not isinstance(values, list) or len(values) != 2:
            continue
        if not isinstance(symbol, str) or symbol not in OPERATIONS:
            continue
        if any(isinstance(v, bool) or not isinstance(v, (int, float)) for v in values):
            continue
        operands = [float(v) for v in values]
        if all(math.isfinite(x) for x in operands):
            return operands, symbol
    raise ValueError(f"No valid operands/operator in output")


def run_syncode(textual_query):
    prompt = _build_prompt(textual_query)
    last_error = None
    for attempt in range(3):
        if attempt == 0:
            active_prompt = prompt
        else:
            reminder = (
                "Output exactly ONE JSON object with two keys, operands and operator. "
                "The operands must be two JSON numbers in calculation order. "
                "Do not output a list, explanation, or the answer.\n\n"
            )
            active_prompt = reminder + prompt
        try:
            generated = syn_llm.infer(active_prompt)
            raw = generated[0] if isinstance(generated, (list, tuple)) else generated
            operands, symbol = _parse_extraction(raw)
            if symbol == "/" and operands[1] == 0.0:
                raise ZeroDivisionError("Division by zero")
            answer = float(OPERATIONS[symbol](operands[0], operands[1]))
            return operands, symbol, answer
        except ZeroDivisionError:
            raise
        except (ValueError, TypeError, KeyError, IndexError) as exc:
            last_error = exc
    raise ValueError(f"SynCode could not extract a valid operation: {last_error}")

## My Tests

In [ ]:
a = run_syncode("What is 10.5 minus 3.25?")
b = run_syncode("What is 327. multiplied by 11.0?")
c = run_syncode("Subtract 8.5 from 20.0.")
d = run_syncode("What is 120.4 divided by 4.0?")
e = run_syncode("What is 45.1 plus 23.54")
print(a,"\n",b,"\n",c,"\n",d,"\n",e)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


([10.5, 3.25], '-', 7.25) 
 ([327.0, 11.0], '*', 3597.0) 
 ([20.0, 8.5], '-', 11.5) 
 ([120.4, 4.0], '/', 30.1) 
 ([45.1, 23.54], '+', 68.64)
